<a href="https://colab.research.google.com/github/Mansik-04/assignments/blob/main/sql%20basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import sqlite3
import pandas as pd

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Helper function to drop tables if they exist
def drop_table_if_exists(table_name):
    drop_query = f"DROP TABLE IF EXISTS {table_name};"
    cursor.execute(drop_query)
    print(f"Dropped table {table_name} if it existed.")

# Drop tables before creating them to ensure a clean run
print("Dropping existing tables...")
drop_table_if_exists("employees")
drop_table_if_exists("products")
drop_table_if_exists("Students")
drop_table_if_exists("Classes")
drop_table_if_exists("Orders")
drop_table_if_exists("Customers")
drop_table_if_exists("Products") # Drop this one again as it's created multiple times
drop_table_if_exists("Sales")
drop_table_if_exists("Products_Sales")
drop_table_if_exists("Orders_Join")
drop_table_if_exists("Customers_Join")
drop_table_if_exists("Order_Details")
print("Finished dropping tables.\n")


# 1. Create the employees table
print("1. Creating the employees table with constraints...")
drop_table_if_exists("employees") # Add drop before creation
create_employees_table_query = """
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY NOT NULL,
    emp_name TEXT NOT NULL,
    age INTEGER CHECK (age >= 18),
    email TEXT UNIQUE,
    salary DECIMAL DEFAULT 30000
);
"""
cursor.execute(create_employees_table_query)
print("employees table created successfully.\n")

# 2. Explanation of constraints
print("2. Explanation of Constraints:")
print("Constraints are rules that enforce data integrity in a database.")
print("- NOT NULL: Ensures a column cannot have a NULL value[cite: 23].")
print("- UNIQUE: Guarantees all values in a column are unique[cite: 18].")
print("- PRIMARY KEY: A unique identifier for each record, which enforces NOT NULL and UNIQUE constraints[cite: 15, 23].")
print("- CHECK: Ensures all values in a column meet a specified condition (e.g., age >= 18)[cite: 17].")
print("- DEFAULT: Provides a default value for a column if none is specified[cite: 19].\n")

# 6. Adding and removing constraints on an existing table
print("6. Adding constraints to the products table...")
# First, create the products table without constraints
drop_table_if_exists("products") # Add drop before creation
create_products_table_query = """
CREATE TABLE products (
    product_id INT,
    product_name VARCHAR(50),
    price DECIMAL(10, 2)
);
"""
cursor.execute(create_products_table_query)
print("products table created without constraints.")

# Add a primary key and a default value
# Note: SQLite's ALTER TABLE has limited support for adding constraints like PRIMARY KEY and DEFAULT.
# The following lines demonstrate the *intent* of adding constraints but are commented out
# because they are not directly supported in SQLite with this syntax.
# To truly add these in SQLite, you'd typically recreate the table.

# print("Attempting to add PRIMARY KEY constraint...")
# add_primary_key_query = """
# ALTER TABLE products
# ADD PRIMARY KEY (product_id);
# """
# cursor.execute(add_primary_key_query)
# print("PRIMARY KEY constraint added to product_id.")

print("The price default of 50.00 would be added with a syntax like: ALTER TABLE products ALTER COLUMN price SET DEFAULT 50.00; (This is standard SQL but not directly supported in SQLite's ALTER TABLE for adding a new default).\n")


# --- Set up tables for remaining questions ---
# 7. Students and Classes tables
print("7. Setting up Students and Classes tables for INNER JOIN...")
drop_table_if_exists("Students") # Add drop before creation
drop_table_if_exists("Classes") # Add drop before creation
students_data = [
    (1, 'Alice', 101),
    (2, 'Bob', 102),
    (3, 'Charlie', 101)
]
classes_data = [
    (101, 'Math'),
    (102, 'Science'),
    (103, 'History')
]
cursor.execute("CREATE TABLE Students (student_id INT, student_name TEXT, class_id INT);")
cursor.executemany("INSERT INTO Students VALUES (?, ?, ?);", students_data)
cursor.execute("CREATE TABLE Classes (class_id INT, class_name TEXT);")
cursor.executemany("INSERT INTO Classes VALUES (?, ?);", classes_data)
conn.commit()

print("Fetching student and class names using INNER JOIN:")
inner_join_query = """
SELECT s.student_name, c.class_name
FROM Students s
INNER JOIN Classes c ON s.class_id = c.class_id;
"""
df_inner_join = pd.read_sql_query(inner_join_query, conn)
print(df_inner_join.to_markdown(index=False))
print("\n")

# 8. Orders, Customers, Products tables for LEFT JOIN
print("8. Setting up tables for LEFT JOIN...")
drop_table_if_exists("Orders") # Add drop before creation
drop_table_if_exists("Customers") # Add drop before creation
drop_table_if_exists("Products") # Add drop before creation
orders_data = [(1, '2024-01-01', 101), (2, '2024-01-03', 102)]
customers_data = [(101, 'Alice'), (102, 'Bob')]
products_data = [(1, 'Laptop', 1), (2, 'Phone', None)]
cursor.execute("CREATE TABLE Orders (order_id INT, order_date TEXT, customer_id INT);")
cursor.executemany("INSERT INTO Orders VALUES (?, ?, ?);", orders_data)
cursor.execute("CREATE TABLE Customers (customer_id INT, customer_name TEXT);")
cursor.executemany("INSERT INTO Customers VALUES (?, ?);", customers_data)
cursor.execute("CREATE TABLE Products (product_id INT, product_name TEXT, order_id INT);")
cursor.executemany("INSERT INTO Products VALUES (?, ?, ?);", products_data)
conn.commit()

print("Fetching all products, even those not associated with an order, using LEFT JOIN:")
left_join_query = """
SELECT o.order_id, c.customer_name, p.product_name
FROM Products p
LEFT JOIN Orders o ON p.order_id = o.order_id
LEFT JOIN Customers c ON o.customer_id = c.customer_id;
"""
df_left_join = pd.read_sql_query(left_join_query, conn)
print(df_left_join.to_markdown(index=False))
print("\n")

# 9. Sales and Products tables for SUM() and GROUP BY
print("9. Setting up tables for SUM() and GROUP BY...")
drop_table_if_exists("Sales") # Add drop before creation
drop_table_if_exists("Products_Sales") # Add drop before creation
sales_data = [(1, 101, 500), (2, 102, 300), (3, 101, 700)]
products_sales_data = [(101, 'Laptop'), (102, 'Phone')]
cursor.execute("CREATE TABLE Sales (sale_id INT, product_id INT, amount INT);")
cursor.executemany("INSERT INTO Sales VALUES (?, ?, ?);", sales_data)
cursor.execute("CREATE TABLE Products_Sales (product_id INT, product_name TEXT);")
cursor.executemany("INSERT INTO Products_Sales VALUES (?, ?);", products_sales_data)
conn.commit()

print("Finding total sales amount for each product:")
sum_group_by_query = """
SELECT ps.product_name, SUM(s.amount) AS total_sales_amount
FROM Sales s
INNER JOIN Products_Sales ps ON s.product_id = ps.product_id
GROUP BY ps.product_name;
"""
df_sum_group_by = pd.read_sql_query(sum_group_by_query, conn)
print(df_sum_group_by.to_markdown(index=False))
print("\n")

# 10. Orders, Customers, Order_Details for multi-table join
print("10. Setting up tables for a multi-table INNER JOIN...")
drop_table_if_exists("Orders_Join") # Add drop before creation
drop_table_if_exists("Customers_Join") # Add drop before creation
drop_table_if_exists("Order_Details") # Add drop before creation
orders_join_data = [(1, '2024-01-02', 1), (2, '2024-01-05', 2)]
customers_join_data = [(1, 'Alice'), (2, 'Bob')]
order_details_join_data = [(1, 101, 2), (1, 102, 1), (2, 101, 3)]
cursor.execute("CREATE TABLE Orders_Join (order_id INT, order_date TEXT, customer_id INT);")
cursor.executemany("INSERT INTO Orders_Join VALUES (?, ?, ?);", orders_join_data)
cursor.execute("CREATE TABLE Customers_Join (customer_id INT, customer_name TEXT);")
cursor.executemany("INSERT INTO Customers_Join VALUES (?, ?);", customers_join_data)
cursor.execute("CREATE TABLE Order_Details (order_id INT, product_id INT, quantity INT);")
cursor.executemany("INSERT INTO Order_Details VALUES (?, ?, ?);", order_details_join_data)
conn.commit()

print("Displaying order details with customer names:")
multi_join_query = """
SELECT o.order_id, c.customer_name, od.quantity
FROM Orders_Join o
INNER JOIN Customers_Join c ON o.customer_id = c.customer_id
INNER JOIN Order_Details od ON o.order_id = od.order_id;
"""
df_multi_join = pd.read_sql_query(multi_join_query, conn)
print(df_multi_join.to_markdown(index=False))
print("\n")

# Close the connection
conn.close()
print("Database connection closed.")

Dropping existing tables...
Dropped table employees if it existed.
Dropped table products if it existed.
Dropped table Students if it existed.
Dropped table Classes if it existed.
Dropped table Orders if it existed.
Dropped table Customers if it existed.
Dropped table Products if it existed.
Dropped table Sales if it existed.
Dropped table Products_Sales if it existed.
Dropped table Orders_Join if it existed.
Dropped table Customers_Join if it existed.
Dropped table Order_Details if it existed.
Finished dropping tables.

1. Creating the employees table with constraints...
Dropped table employees if it existed.
employees table created successfully.

2. Explanation of Constraints:
Constraints are rules that enforce data integrity in a database.
- NOT NULL: Ensures a column cannot have a NULL value[cite: 23].
- UNIQUE: Guarantees all values in a column are unique[cite: 18].
- PRIMARY KEY: A unique identifier for each record, which enforces NOT NULL and UNIQUE constraints[cite: 15, 23].
- 